# Proyecto Final — Construccion de un Compilador
## GoldenRetrieverLang (GRL)

**Curso:** Compiladores  
**Valor total del proyecto:** 30%  
**Lenguaje de implementacion:** C (con Flex + Bison)  
**Referencia base:** Aho, Lam, Sethi y Ullman (2008). *Compiladores: principios, tecnicas y herramientas*. Pearson Educacion.

Este notebook documenta y demuestra el compilador/interprete **GoldenRetrieverLang** en sus cuatro fases clasicas: lexica, sintactica, semantica y de ejecucion. Ademas incluye celdas ejecutables que compilan el binario `golden` y lo prueban con los archivos fuente del repositorio.

## 1. Resumen Ejecutivo

GoldenRetrieverLang (GRL) es un lenguaje academico de tipado estatico con sintaxis en espanol tematica de Golden Retriever. El compilador se implementa en C siguiendo las fases clasicas:

1. **Analisis lexico** `lexer.l` (Flex).
2. **Analisis sintactico** `parser.y` (Bison) con construccion de un **AST**.
3. **Analisis semantico** `src/symbols.c`, `src/parser_helper.c` con tabla de simbolos basada en **pila de ambitos** y **shadowing** lexical.
4. **Generacion/Ejecucion** `src/interpreter.c` mediante un interprete de arbol (*tree-walking*).

El entregable es el binario `golden`, que recibe un archivo fuente `.grl` y ejecuta el programa validando sintaxis, tipos y ambitos.

## 2. Propuesta del Lenguaje

### 2.1 Tipos de datos

| Palabra clave | Tipo interno | Descripcion |
|---------------|--------------|-------------|
| `HUESO`       | `int`        | Entero con signo |
| `CORREA`      | `char*`      | Cadena de texto (string) |

### 2.2 Palabras reservadas

`HUESO`, `CORREA`, `LADRA`, `RETORNA`, `IF`, `ELIF`, `ELSE`, `SI`, `SINO_SI`, `SINO`, `OLFATEA_HUESO`, `OLFATEA_CORREA`.

### 2.3 Operadores

- Aritmeticos enteros: `+`, `-`, `*`, `/` (division entera).
- Concatenacion de strings: `+`.
- Comparacion (solo enteros, retornan `HUESO` 0/1): `==`, `!=`, `<`, `<=`, `>`, `>=`.
- Asignacion: `=`.
- Agrupacion: `(` `)`, bloques `{` `}`.

### 2.4 Gramatica simplificada (EBNF)

```ebnf
programa      := { sentencia } EOF ;
sentencia     := decl_var | asignacion | print | condicional
               | bloque | decl_func | retorno ;
decl_var      := ("HUESO" | "CORREA") ID "=" expr ";" ;
asignacion    := ID "=" expr ";" ;
print         := "LADRA" "(" expr ")" ";" ;
condicional   := (IF|SI) "(" expr ")" bloque
                 { (ELIF|SINO_SI) "(" expr ")" bloque }
                 [ (ELSE|SINO) bloque ] ;
bloque        := "{" { sentencia } "}" ;
decl_func     := ("HUESO"|"CORREA") ID "(" [ parametros ] ")" bloque ;
parametros    := param_decl { "," param_decl } ;
param_decl    := ("HUESO"|"CORREA") ID ;
retorno       := "RETORNA" expr ";" ;
expr          := sum { comparador sum } ;
sum           := term { ("+" | "-") term } ;
term          := factor { ("*" | "/") factor } ;
factor        := NUMBER | STRING | ID | llamada | entrada | "(" expr ")" ;
llamada       := ID "(" [ expr { "," expr } ] ")" ;
entrada       := "OLFATEA_HUESO" "(" ")" | "OLFATEA_CORREA" "(" ")" ;
```

## 3. Implementacion de las Fases (Rubrica I — 10%)

### 3.1 Analisis Lexico — `lexer.l`

Se utilizo **Flex** para generar el analizador lexico. El archivo `lexer.l` define clases de caracteres, patrones y acciones que convierten el flujo de caracteres de entrada en **tokens** reconocidos por el parser.

**Aspectos destacables:**

- Tokens para palabras reservadas (`HUESO`, `CORREA`, `LADRA`, `RETORNA`, `IF/ELIF/ELSE`, alias espanoles `SI/SINO_SI/SINO`, entradas de teclado).
- Identificadores con expresion regular `[A-Za-z_][A-Za-z0-9_]*`.
- Numeros enteros con `atoi`.
- Strings con soporte de secuencias de escape `\n`, `\t`, `\"` (funcion `unescape_string`).
- Comentarios de linea `// ...` y bloque `/* ... */`.
- Operadores relacionales multicarácter antes que los de un solo caracter.
- `%option yylineno` para reportes de error con numero de linea.

In [ ]:
# Mostrar el archivo lexer.l (primeras 60 lineas)
!sed -n '1,60p' lexer.l

### 3.2 Analisis Sintactico — `parser.y`

El parser se implementa con **Bison** (LALR(1)). Se declaran precedencias para resolver ambiguedad de la gramatica de expresiones:

```
%left EQ NE LT LE GT GE
%left '+' '-'
%left '*' '/'
```

Durante el analisis, **cada regla construye nodos del AST** usando los constructores definidos en `src/ast.c` (`ast_make_*`). El AST resultante se almacena en `g_root` dentro de `parser.y` y es recorrido por el interprete cuando el parseo termina con exito.

**Tipos de nodo del AST** (`src/headers/ast.h`):

```
AST_BLOCK, AST_SCOPE, AST_VAR_DECL, AST_ASSIGN, AST_PRINT, AST_IF,
AST_INPUT, AST_FUNC_DECL, AST_RETURN, AST_INT, AST_STRING, AST_VAR,
AST_BINOP, AST_FUNC_CALL
```

In [ ]:
# Mostrar las reglas del parser (lineas relevantes)
!sed -n '42,120p' parser.y

### 3.3 Analisis Semantico — `src/symbols.c`, `src/parser_helper.c`

La tabla de simbolos se modela como una **pila de ambitos** (`scope_stack` con `MAX_SCOPES = 64` y `MAX_SYMBOLS_PER_SCOPE = 256`). Cada entrada almacena nombre, valor (`Value`), tipo y un `offset` incremental que emula la asignacion de memoria local.

**Reglas semanticas verificadas:**

| Regla | Mecanismo |
|-------|-----------|
| No re-declarar variable en el mismo ambito | `declare_symbol` devuelve 0 si ya existe |
| Una variable debe estar declarada antes de usarse | `variable_value_checked` → `helper_fail` |
| `HUESO` solo guarda enteros, `CORREA` solo strings | Cotejo de `expected_type` en declaracion |
| Reasignacion preserva el tipo original | `assign_symbol` compara tipos |
| `+` permitido entre enteros y entre strings | `add_values_checked` |
| `-`, `*`, `/` exclusivos de enteros | `sub/mul/div_values_checked` |
| Division por cero | `div_values_checked` → error en linea |
| Comparacion solo entre enteros | `eval_compare` |
| Condicion de `IF` debe ser `HUESO` | `exec_if` valida tipo |
| Argumentos coinciden en numero/tipo | `eval_call` en `interpreter.c` |
| `RETORNA` solo dentro de funciones | Flag `in_function` |
| Shadowing por bloque | `push_scope` / `pop_scope` |

In [ ]:
# Resumen de la pila de ambitos
!sed -n '93,170p' src/symbols.c

### 3.4 Generacion / Ejecucion — `src/interpreter.c`

Se opto por un **interprete de AST** (*tree-walking interpreter*) en lugar de generacion a codigo maquina o ensamblador, decision justificada por el alcance academico y la posibilidad de demostrar todas las fases semanticas en tiempo real.

**Flujo de ejecucion `execute_ast`:**

1. Registrar todas las funciones declaradas (`register_all_functions`) para permitir llamadas hacia adelante.
2. Recorrer las sentencias de nivel superior del AST, omitiendo declaraciones de funcion.
3. Para cada sentencia, invocar `exec_statement`, que despacha por `node->type` a la rutina correspondiente.
4. Las expresiones se evaluan en `eval_expr`, que produce un `Value` (entero o string) correctamente liberado tras su uso.
5. La llamada a funcion (`eval_call`) abre un scope nuevo, enlaza parametros, ejecuta el cuerpo y recupera el valor via `ExecResult` (`has_return`, `value`).

## 4. Funcionamiento y Pruebas (Rubrica II — 10%)

### 4.1 Compilacion

La proxima celda ejecuta `make clean && make` para reconstruir el binario `golden`.

In [ ]:
!make clean && make

### 4.2 Prueba 1 — Shadowing de 3 niveles (`prueba`)

Este caso valida la correcion de la pila de ambitos. Un mismo identificador (`energia`, `estado`) se declara en 4 niveles y al salir de cada bloque el valor visible vuelve al nivel externo.

Salida esperada (con entrada `Luna\n7\n`):

```
100
Max
70
Patio
40
70
10
Parque
40
Patio
70
100
Luna
7
```

In [ ]:
!printf 'Luna\n7\n' | ./golden prueba

In [ ]:
# Comparacion estricta contra la salida esperada
!make test

### 4.3 Prueba 2 — `calculadora.grl`

Pide dos enteros por teclado, calcula suma/resta/multiplicacion y usa `IF/ELSE` para proteger la division por cero.

In [ ]:
!cat calculadora.grl

In [ ]:
!printf '10\n4\n' | ./golden calculadora.grl

### 4.4 Prueba 3 — `calculadora2.grl`

Menu con `ELIF` anidados e `IF` interno para manejar division por cero.

In [ ]:
!cat calculadora2.grl

In [ ]:
# Ejemplo: opcion 1 (suma) con 8 + 7
!printf '1\n8\n7\n' | ./golden calculadora2.grl

In [ ]:
# Ejemplo: opcion 3 (division) con divisor cero -> mensaje controlado
!printf '3\n10\n0\n' | ./golden calculadora2.grl

### 4.5 Manejo de errores (Robustez)

Todos los errores se canalizan por `helper_fail` con numero de linea:

```c
void helper_fail(const char* msg) {
    fprintf(stderr, "Error en linea %d: %s\n", yylineno, msg);
    cleanup_symbols();
    exit(1);
}
```

Mensajes especificos: *Variable no declarada*, *Declaracion invalida o variable repetida*, *Suma entre tipos incompatibles*, *Division entre cero*, *Comparacion solo para enteros*, *Funcion no declarada*, *Cantidad de argumentos invalida*, *RETORNA fuera de una funcion*.

In [ ]:
# Demostracion de error semantico: tipo incompatible
%%bash
cat > /tmp/err_tipo.grl <<'EOF'
HUESO x = "texto";
EOF
./golden /tmp/err_tipo.grl; echo "--- fin (codigo $?)"


In [ ]:
# Demostracion de error: variable no declarada
%%bash
cat > /tmp/err_var.grl <<'EOF'
LADRA(x);
EOF
./golden /tmp/err_var.grl; echo "--- fin (codigo $?)"


In [ ]:
# Demostracion de error: division por cero
%%bash
cat > /tmp/err_div.grl <<'EOF'
HUESO a = 10;
HUESO b = 0;
LADRA(a / b);
EOF
./golden /tmp/err_div.grl; echo "--- fin (codigo $?)"


## 5. Defensa y Sustentacion (Rubrica III — 10%)

### 5.1 Dominio tecnico — puntos clave

- **Por que Flex + Bison?** Son herramientas estandar derivadas de Lex/Yacc descritas en el libro de Aho. Generan DFA y parser LALR(1) automaticamente.
- **Por que interprete de AST y no codegen?** El alcance del curso privilegia claridad de las fases. El AST hace visible el analisis sintactico y semantico.
- **Como se maneja el shadowing?** Con pila de ambitos. La busqueda de un identificador se detiene en el primer nivel de arriba hacia abajo que lo contenga (`find_in_stack`).
- **Por que un offset por variable si no se emite ensamblador?** Documenta el modelo de activacion y deja el camino abierto para una fase futura de generacion de codigo.
- **Llamadas hacia adelante** Se hace un pase previo (`register_all_functions`) que registra cada `AST_FUNC_DECL` antes de ejecutar.
- **RETORNA fuera de funcion** Se propaga `has_return` en `ExecResult`; `execute_ast` detecta el caso y dispara `helper_fail`.

### 5.2 Organizacion del codigo

```
.
├── lexer.l                 # Analisis lexico (Flex)
├── parser.y                # Analisis sintactico + construccion AST (Bison)
├── src/
│   ├── ast.c               # Constructores y liberacion del AST
│   ├── interpreter.c       # Interprete tree-walking + funciones
│   ├── parser_helper.c     # Errores y aritmetica con chequeo de tipos
│   ├── symbols.c           # Pila de ambitos y tabla de simbolos
│   └── headers/
│       ├── ast.h
│       ├── interpreter.h
│       ├── parser_helper.h
│       └── symbols.h
├── Makefile                # Targets: all, run, test, clean
├── prueba                  # Prueba de shadowing (3 niveles)
├── calculadora.grl         # Calculadora basica
├── calculadora2.grl        # Calculadora con menu
├── ReglasLenguaje.L        # Especificacion del lenguaje
└── README.md               # Guia de uso
```

In [ ]:
!ls -la

### 5.3 Decisiones de diseno justificadas

| Decision | Justificacion |
|----------|---------------|
| C11 + Flex + Bison | Cobertura directa del contenido de Aho et al. |
| Tipado estatico con 2 tipos | Fuerza al parser a razonar sobre tipos sin complicar. |
| AST como IR principal | Permite separar limpiamente sintaxis y ejecucion. |
| Pila de ambitos en arreglo fijo | O(1) de push/pop y codigo sencillo. |
| Interprete vs codegen | Prioriza demostrar semantica; extensible a codegen. |
| Mensajes con `yylineno` | Localizacion util para el usuario final. |

### 5.4 Limitaciones reconocidas

- No hay bucles (`WHILE` / `FOR`); el lenguaje cubre condicionales y funciones con recursion.
- La division es entera; no hay `float`.
- Concatenacion solo entre strings; no hay conversion implicita entero-string.
- Maximo 64 niveles de ambito y 256 simbolos por nivel (ajustables en `symbols.c`).

### 5.5 Posibles extensiones futuras

1. Generacion a codigo intermedio de tres direcciones (TAC) y luego a ensamblador x86-64 o WebAssembly.
2. Tipos adicionales (booleano explicito, flotante).
3. Bucles `MIENTRAS` / `PARA` y `ROMPE` / `CONTINUA`.
4. Optimizaciones sobre el AST: plegado de constantes, eliminacion de codigo muerto.

## 6. Conclusiones

El proyecto cubre las cuatro fases clasicas de un compilador y produce un sistema funcional capaz de analizar, validar y ejecutar programas escritos en GoldenRetrieverLang. La separacion clara entre `lexer.l`, `parser.y`, la tabla de simbolos y el interprete hace que cada fase sea verificable y defendible de manera independiente.

La prueba de shadowing (`prueba`) y las calculadoras (`calculadora*.grl`) evidencian que el analizador lexico, el parser, el chequeo de tipos y la gestion de ambitos funcionan en conjunto de forma estable y predecible, cumpliendo con los criterios de *Funcionalidad Completa y Estabilidad*, *Manejo de Errores* y *Ejecucion y Resultados* de la rubrica.

## 7. Referencias

- Aho, A., Lam, M., Sethi, R. y Ullman, J. (2008). *Compiladores: principios, tecnicas y herramientas* (2a ed.). Pearson Educacion.
- Flex manual — https://westes.github.io/flex/manual/
- Bison manual — https://www.gnu.org/software/bison/manual/
- Compiler Tools catalog — http://catalog.compilertools.net/